In [ ]:
USE WAREHOUSE DV_COMPUTE_WH;

USE DATABASE LEAGUE_RECORDS;

USE SCHEMA L30_ID;


In [ ]:
from typing import Callable, Annotated
from enum import Enum

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pydantic import (
    BaseModel,
    field_validator,
    model_validator,
    Field
)
from scipy.stats import norm

# Analyzing relationship between kills and CS
* Explanatory variable: net_CS (CS + Jungle CS)
* Response variable: kills

In [ ]:
SELECT 
    MINUTE, 
    CS,
    JUNGLE_CS,
    CS + JUNGLE_CS AS NET_CS,
    KILLS, 
    DEATHS, 
    ASSISTS, 
    COALESCE(ROUND(
        (KILLS + ASSISTS) / NULLIF(DEATHS, 0)
    , 2), KILLS + ASSISTS) AS KDA_RATIO
FROM FCT_INTERVALS
JOIN DIM_INTERVALS_CS USING(PLAYER_INTERVAL_ID)
JOIN DIM_INTERVALS_KDA USING(PLAYER_INTERVAL_ID)
ORDER BY MINUTE ASC
;

In [ ]:
def sample_dataset(
    df: pd.DataFrame, 
    fixed_sampling: int = None,
    pct_sampling: float = None,
    seed: int = None
) -> pd.DataFrame:
    sample_size = None
    if fixed_sampling is not None:
        sample_size = fixed_sampling
    elif pct_sampling is not None:
        sample_size = int(len(df) * pct_sampling)
    else:
        sample_size = 100
        
    return df.sample(sample_size, random_state=seed)

In [ ]:
def visualize_scatterplot(data: pd.DataFrame, x_col: str, y_col: str, **kwargs) -> None:
    fig, ax = plt.subplots()
    sns.scatterplot(
        data=data,  # ← use the parameter
        x=x_col,
        y=y_col,
        ax=ax,
        **kwargs
    )
    plt.show()
    
    plt.close(fig) 

In [ ]:
def bivariate_rls(
    explanatory: str,
    response: str
) -> None:
    sample = sample_dataset(
        cs_vs_kda, 
        fixed_sampling=100,
    )
    visualize_scatterplot(
        sample, 
        x_col=explanatory,
        y_col=response
    )
    print(len(sample))
    

In [ ]:
bivariate_rls(
    explanatory='ASSISTS',
    response='KILLS'
)

# Probability Questions
1. What is the probability of a given player having over x kills at x minute.

In [ ]:
class LeagueMatchInterval(BaseModel):
    min_interval: Annotated[int, Field(..., ge=5, le=90)]

    @field_validator('min_interval', mode='before')
    @classmethod
    def factor_of_5(cls, v: int) -> int:
        if v % 5 != 0:
            raise ValueError(f'min_interval must be a multiple of 5, got {v}')
        
        return v

In [ ]:
def stats_at_minute(
    df: pd.DataFrame, 
    stats: list[str],
    minute: int,
) -> pd.DataFrame:
    minute = LeagueMatchInterval(min_interval=minute).min_interval
    
    return df.loc[
        df['MINUTE'] == minute, 
        ['MINUTE'] + stats
    ]

In [ ]:
def scale_sqrt_var(df: pd.DataFrame, col: str) -> pd.DataFrame:
    return df.assign(
        **{f'sqrt_{col}': np.sqrt(df[col])}
        )

In [ ]:
def visualize_histogram(
    data: pd.DataFrame, 
    x_col: str, 
    **kwargs
) -> None:
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    sns.histplot(
        data=data, 
        x=x_col,
        ax=ax,
        color='#FF9999',
        kde=True,
        **kwargs
    )
    
    ax.set_title(f'Distribution of {x_col.replace("_", " ").title()}')
    ax.set_xlabel(x_col.replace('_', ' ').title())
    ax.set_ylabel('Count')
    
    plt.show()
    plt.close(fig)

In [ ]:
def calc_probability(
    mean: float, 
    std: float, 
    bound: float
) -> float:
    return np.round(
        norm.cdf(bound, loc=mean, scale=std) * 100
    , 2)

In [ ]:
def probability_of_kills_at_minute(
    kills: int,
    minute: int,
    visualizing_kwargs: dict = {}
) -> None:
    kills_at_end = stats_at_minute(
        cs_vs_kda, 
        stats=['KILLS'],
        minute=minute
    )
    sample = sample_dataset(
        kills_at_end, 
        pct_sampling=0.1,
    )
    adjusted_sample = scale_sqrt_var(sample, 'KILLS')
    print(f"Sample size --> {len(sample)}")

    visualize_histogram(
        data=adjusted_sample,
        x_col='sqrt_KILLS',
        **visualizing_kwargs
    )

    mean_kill_of_sample = np.mean(adjusted_sample['sqrt_KILLS'])
    std_kill_of_sample = np.std(adjusted_sample['sqrt_KILLS'], ddof=1)
    print(f"Mean --> {mean_kill_of_sample}")
    print(f"Standard Deviation --> {std_kill_of_sample}")

    p_kill_at_min = calc_probability(
        mean=mean_kill_of_sample,
        std=std_kill_of_sample,
        bound=kills ** 0.5
    )
    
    print(f"At minute {minute}, you have a {p_kill_at_min}% of having already scored {kills} kills")
    return p_kill_at_min


In [ ]:
def sample_distribution() -> None:
    prob_array = []
    for _ in range(10):
        result = probability_of_kills_at_minute(
            kills=10, 
            minute=30,
            visualizing_kwargs={'bins': 20}
        )
        prob_array.append(result)

    sns.histplot(prob_array)
    
    plt.show()
    print(f"Final result: {np.mean(prob_array)}")